# **Phase 1**

In [40]:
# Imports 1 

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_openai import ChatOpenAI

In [ ]:
# model intro 2

from pydantic import BaseModel, Field

class FitnessAnalysis(BaseModel):
    observation: str = Field(description="1-2 short lines observing the image.")
    analysis: str = Field(description="1-2 short lines analyzing the fitness aspect.")
    suggestions: list[str] = Field(description="2-3 bullet points of suggestions.")
    tips: list[str] = Field(description="1-2 short fitness tips.")

In [ ]:
# model 3

import langchain
from langchain_community.cache import SQLiteCache

langchain.llm_cache = SQLiteCache(database_path="langcahin_cache.db")

model = ChatOpenAI(model="gpt-4o-mini").with_structured_output(FitnessAnalysis)

In [52]:
# prompt template 4

from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an expert, friendly personal trainer. Your goal is to give clear, actionable advice based on what you see in the user's image and their query.

COMPANY FITNESS PROTOCOLS (CRITICAL):
Below are the strict proprietary rules for our startup. You MUST prioritize these rules in your advice if they apply to the user's situation.
{context}

CRITICAL INSTRUCTION: Avoid overly dense medical jargon. Sound encouraging but direct.

Structure your exact response using this strict JSON schema layout:
Observation: (What you clearly see in the image regarding their posture, form, or physique relative to their query. Keep it simple.)
Analysis: (Why this happens in plain English.)
Suggestions: (Give 2-3 specific exercises or stretches. ALWAYS apply the Company Fitness Protocols if relevant!)
Tips: (1-2 quick, practical form cues or daily habits.)

Safety Constraints:
- Be highly supportive and motivating.
- Never diagnose medical conditions.
"""),

    ("human", [
        {"type": "text", "text": "{query}"},
        {
            "type": "image_url",
            "image_url": {
                "url": "{image_url}",
                "detail": "low"
            }
        }
    ])
])

In [ ]:
# image upload 5

import hashlib
import json
import os
from PIL import Image
import cloudinary
import cloudinary.uploader

from dotenv import load_dotenv
import os

# Force load_dotenv to look at the exact file in the current folder
load_dotenv(dotenv_path=os.path.join(os.getcwd(), ".env"))

REGISTRY_FILE = "image_registry.json"

# Load existing cache from disk if it exists, otherwise start fresh
if os.path.exists(REGISTRY_FILE):
    with open(REGISTRY_FILE, "r") as f:
        IMAGE_REGISTRY = json.load(f)
else:
    IMAGE_REGISTRY = {}

cloudinary.config(
    cloud_name=os.getenv("CLOUDINARY_CLOUD_NAME"),
    api_key=os.getenv("CLOUDINARY_API_KEY"),
    api_secret=os.getenv("CLOUDINARY_API_SECRET")
)

def calculate_image_hash(file_path):
    hasher = hashlib.md5()
    with open(file_path, "rb") as f:
        for chunk in iter(lambda: f.read(4096), b""):
            hasher.update(chunk)
    return hasher.hexdigest()

def upload_image(image_path):
    img_hash = calculate_image_hash(image_path)
    
    if img_hash in IMAGE_REGISTRY:
        print(" -> [Persistent Image Cache Hit] Reusing stored Cloudinary URL!")
        return IMAGE_REGISTRY[img_hash]
        
    print(" -> [Image Cache Miss] New image. Optimizing and uploading...")
    
    img = Image.open(image_path)
    img = img.convert("RGB")
    img = img.resize((320, 320))

    temp_path = "temp.jpg"
    img.save(temp_path, quality=60)

    response = cloudinary.uploader.upload(temp_path, resource_type="image")
    secure_url = response["secure_url"]
    
    # Save to memory and immediately write to the JSON file
    IMAGE_REGISTRY[img_hash] = secure_url
    with open(REGISTRY_FILE, "w") as f:
        json.dump(IMAGE_REGISTRY, f, indent=2)
    
    return secure_url

In [ ]:
# vector db setup 6

import os
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from dotenv import load_dotenv

load_dotenv(dotenv_path=os.path.join(os.getcwd(), ".env"))

# 1. Define where to store our local vector database files
CHROMA_PATH = "chroma_db"
DOCUMENT_PATH = "Rules_dummy.txt"

print("Initializing Vector Database...")

# 2. Check if the database already exists to avoid re-building it every time
if os.path.exists(CHROMA_PATH):
    print(" -> [Vector DB] Found existing ChromaDB database. Loading it...")
    db = Chroma(
        persist_directory=CHROMA_PATH, 
        embedding_function=OpenAIEmbeddings()
    )
else:
    print(" -> [Vector DB] Database not found. Creating a fresh one from text file...")
    
    # Load your mock fitness guidelines file
    if not os.path.exists(DOCUMENT_PATH):
        raise FileNotFoundError(f"Please create the file: {DOCUMENT_PATH}")
        
    with open(DOCUMENT_PATH, "r") as f:
        raw_text = f.read()
    
    # Standard splitting method: Split by double newlines or single paragraphs
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=400,
        chunk_overlap=50,
        separators=["\n\n", "\n", " "]
    )
    
    # Turn raw text strings into document chunks
    chunks = text_splitter.create_documents([raw_text])
    
    # 3. Create the database and save it to disk
    db = Chroma.from_documents(
        documents=chunks,
        embedding=OpenAIEmbeddings(),
        persist_directory=CHROMA_PATH
    )
    print(f" -> [Vector DB] Success! Loaded {len(chunks)} chunks into ChromaDB.")

# 4. Turn the database into a 'Retriever' tool that can search for matches
retriever = db.as_retriever(search_kwargs={"k": 2}) # Pulls top 2 most relevant chunks

Initializing Vector Database...
 -> [Vector DB] Database not found. Creating a fresh one from text file...
 -> [Vector DB] Success! Loaded 3 chunks into ChromaDB.


In [ ]:
# input validation 6

def validate_input(data):
    query = data["query"].lower()
    
    blocked_words = ["who is", "identify", "celebrity", "name", "rate me"]

    if any(word in query for word in blocked_words):
        return {
            "query": "This system only supports fitness-related analysis.",
            "image": data["image_url"]
            }
    
    return data

In [ ]:
# runnable setup 7

validator = RunnableLambda(validate_input)

In [53]:
# chain 8

chain = validator | prompt | model

In [54]:
# execution 9

from langchain_community.callbacks import get_openai_callback

image_path = input("Enter image path: ")
user_query = input("Enter your query: ")

try:
    image_url = upload_image(image_path)
    print("Image uploaded successfully!")

    # --- NEW RAG STEP ---
    # 1. Search ChromaDB for rules matching the user's query
    print(" -> Searching proprietary company protocols...")
    docs = retriever.invoke(user_query)
    
    # 2. Combine the matched text chunks into one string
    rag_context = "\n\n".join(doc.page_content for doc in docs)
    # --------------------

    with get_openai_callback() as cb:
        # 3. Pass the 'context' into the chain alongside the query and image
        response = chain.invoke({
            "query": user_query,
            "image_url": image_url,
            "context": rag_context
        })

    output = response

    print("\n--- Fitness Analysis ---\n")
    print(output.model_dump_json(indent=2))

    print("\n--- Token Usage ---")
    print(f"Prompt Tokens: {cb.prompt_tokens}")
    print(f"Comletion Tokens: {cb.completion_tokens}")
    print(f"Total Tokens: {cb.total_tokens}")

    usd_cost=cb.total_cost
    inr_cost=usd_cost*95.60
    print(f"Total cost (inr): {inr_cost:.4f}")

except Exception as e:
    print("Error:", e)



 -> [Persistent Image Cache Hit] Reusing stored Cloudinary URL!
Image uploaded successfully!
 -> Searching proprietary company protocols...

--- Fitness Analysis ---

{
  "observation": "You have a lean physique with some abdominal fat.",
  "analysis": "This suggests that you're at a stage where focusing on core stabilization and nutrition could be more effective than fat burners.",
  "suggestions": [
    "Incorporate 3 sets of 45-second planks to strengthen your core.",
    "Add 3 sets of 15 hanging leg raises to target your lower abs.",
    "Consider a slight caloric surplus of +150 calories with an emphasis on lean protein."
  ],
  "tips": [
    "Maintain good posture during exercises to engage your core effectively.",
    "Stay consistent with your workouts and nutrition for the best results."
  ]
}

--- Token Usage ---
Prompt Tokens: 3321
Comletion Tokens: 119
Total Tokens: 3440
Total cost (inr): 0.0544
